# 08_governanca_metadados
Enriquece os metadados e aplica a governança no Unity Catalog: comentários de tabela e coluna, tags (camada, domínio, PII),
chaves primárias e estrangeiras (informativas), máscara de coluna nos dados pessoais da Bronze e concessões de acesso.

Idempotente: pode ser executado depois de cada execução do pipeline. Cada comando é tentado separadamente e o resumo final lista o que falhou.

In [ ]:
import json
from datetime import datetime

CATALOGO = "desafio_vendas"
GRUPO_PII = "pii_autorizados"          # grupo com acesso ao dado pessoal em claro (se existir no seu ambiente)
EXECUTAR_GRANTS = False                # True para conceder leitura da Gold ao grupo "account users"
DONO = spark.sql("SELECT current_user()").first()[0]   # quem executa o pipeline continua vendo o dado em claro

resultado = []
def sql(cmd, rotulo):
    try:
        spark.sql(cmd)
        resultado.append((rotulo, "OK", ""))
    except Exception as e:
        msg = str(e).replace("\n", " ")[:160]
        ja_existe = "already exists" in msg.lower() or "ALREADY_EXISTS" in msg
        resultado.append((rotulo, "OK (já existia)" if ja_existe else "FALHA", "" if ja_existe else msg))

## 1. Comentários de catálogo, schemas, tabelas e colunas

In [ ]:
COMENTARIOS_TABELAS = json.loads(r'''{"bronze.vendas_raw": "Dado bruto, fiel ao CSV (tudo string), com colunas de auditoria e nomes de colunas padronizados.", "silver.vendas": "Vendas tipadas, padronizadas e sem dados pessoais em claro. Carrega a lista de alertas de qualidade por registro.", "silver.vendas_quarentena": "Registros da Bronze rejeitados: erro de qualidade (3.959) ou duplicata (1.000), com a regra violada.", "gold.dim_tempo": "Dimensão calendário (01/12/2020 a 31/03/2024). Usada nos papéis de data da venda e data da retirada.", "gold.dim_produto": "Dimensão produto e tipo de produto.", "gold.dim_localidade": "Dimensão localidade: cidade, UF e região. Valores ausentes viram \"Não informada\" / \"NI\".", "gold.dim_pagamento": "Dimensão forma de pagamento.", "gold.dim_status": "Dimensão junk: combinação de status da venda e status do envio.", "gold.dim_cliente": "Dimensão cliente, sem dados pessoais. Sexo = valor mais frequente do cliente.", "gold.fato_vendas": "Fato de vendas: quantidade, valor bruto, desconto, valor líquido, frete, flags de negócio e chaves das dimensões.", "gold.agg_vendas_mensal_produto": "Agregação mensal por produto para BI e para o modelo de previsão. Receita = aprovadas; demanda = aprovadas + pendentes.", "gold.vw_vendas_analitica": "View desnormalizada (fato + dimensões) para BI e ciência de dados. Sem dados pessoais.", "gold.previsao_demanda_mensal": "Previsão de demanda mensal (jan a mar/2024) com intervalo de 80%.", "gold.recomendacao_estoque": "Necessidade de estoque para cobrir 3 meses: demanda prevista + estoque de segurança (nível de serviço de 95%). Não considera saldo atual.", "gold.backtest_modelos": "Resultado do backtest por modelo (MAE, RMSE, WAPE, viés) e modelo escolhido.", "ops.log_execucao": "Log do orquestrador: status, duração, contagens, lote da camada e mensagem do gate.", "ops.log_pipeline": "Volumes lidos, gravados, rejeitados e duplicados por camada.", "ops.qualidade_resultados": "Resultado de cada regra de qualidade (Silver) e verificação (Gold).", "ops.vw_ultima_execucao": "Etapas da execução mais recente do orquestrador.", "ops.vw_historico_execucoes": "Resumo por execução: início, fim, duração de relógio e status.", "ops.vw_funil_registros": "Último volume por camada: lidos, gravados, rejeitados, duplicados.", "ops.vw_qualidade_ultima": "Última medição de cada regra, com situação (OK, QUARENTENA, ALERTA, CRITICO).", "ops.vw_score_qualidade": "Score: % de registros sem alertas, sem alertas exceto R08, e % de retirada inconsistente."}''')
COMENTARIOS_COLUNAS = json.loads(r'''{"bronze.vendas_raw": {"produto": "Produto vendido, com erros de digitação do arquivo original. Origem/regra: Cópia de \"Produto\"", "data_venda": "Data da venda como veio (3 formatos misturados). Origem/regra: Cópia de \"Data da venda\"", "data_retirada": "Data de retirada como veio (3 formatos misturados). Origem/regra: Cópia de \"Data da retirada\"", "valor_frete": "Valor do frete. Origem/regra: Cópia de \"Valor do frete\"", "status_envio": "Status do envio (variações de caixa e espaços). Origem/regra: Cópia de \"Status de envio\"", "cidade": "Cidade (variações de caixa; possui nulos). Origem/regra: Cópia de \"Cidade\"", "estado": "Estado como veio (siglas, nomes por extenso e erros). Origem/regra: Cópia de \"Estado\"", "cliente": "Identificador do cliente (\"Cliente NNNNN\"). Origem/regra: Cópia de \"Cliente\"", "cliente_cpf": "CPF do cliente (PII; formatos inválidos presentes). Origem/regra: Cópia de \"CPF do cliente\"", "cliente_sexo": "Sexo do cliente (M, F, Outro, 1, 2). Origem/regra: Cópia de \"Sexo do cliente\"", "tipo_produto": "Categoria do produto. Origem/regra: Cópia de \"Tipo de produto\"", "qtd_vendida": "Quantidade vendida. Origem/regra: Cópia de \"Quantidade vendida\"", "valor_unitario": "Preço unitário. Origem/regra: Cópia de \"Preço unitário\"", "status_venda": "Status da venda. Origem/regra: Cópia de \"Status da venda\"", "tipo_pagamento": "Forma de pagamento. Origem/regra: Cópia de \"Tipo de pagamento\"", "avaliacao_produto": "Nota do produto. Origem/regra: Cópia de \"Avaliação do produto\"", "status_desconto": "Indica se houve desconto (Sim/Não). Origem/regra: Cópia de \"Status de desconto\"", "desconto_pct": "Percentual de desconto. Origem/regra: Cópia de \"Porcentagem de desconto\"", "item_promocional": "Indica item promocional (Sim/Não). Origem/regra: Cópia de \"Item promocional\"", "cliente_telefone": "Telefone de contato (PII). Origem/regra: Cópia de \"Telefone de contato\"", "cliente_email": "E-mail de contato (PII). Origem/regra: Cópia de \"E-mail de contato\"", "_arquivo_origem": "Nome do arquivo de origem. Origem/regra: Constante: Dataset_vendas_1.csv", "_data_ingestao": "Momento da ingestão. Origem/regra: current_timestamp()", "_id_lote": "Identificador (UUID) da execução da Bronze. Origem/regra: uuid4 gerado a cada execução", "_numero_linha_origem": "Identificador único e crescente da linha na ingestão. Origem/regra: monotonically_increasing_id() (único; não é sequencial contínuo)"}, "silver.vendas": {"id_venda": "Identificador da venda na Silver. Origem/regra: row_number() por _numero_linha_origem, só registros válidos", "produto": "Produto padronizado (Notebook, Smartphone, Smartwatch). Origem/regra: Dicionário sobre trim+lower; corrige Not-book, Smart phon, Smart watchi", "tipo_produto": "Categoria (Informática, Eletrônico, Acessório). Origem/regra: Cópia da Bronze; regra R19 confere coerência com o produto", "data_venda": "Data da venda. Origem/regra: try_to_date em yyyy-MM-dd, dd/MM/yyyy e MM-dd-yyyy; ausente ou inválida = erro R01", "data_retirada": "Data de retirada. Origem/regra: try_to_date em yyyy-MM-dd, yyyy/MM/dd e dd-MM-yyyy; inválida = alerta R06", "valor_frete": "Valor do frete em R$. Origem/regra: try_cast; ausente permanece NULL (R10), sem imputação", "status_envio": "Status do envio padronizado. Origem/regra: Dicionário sobre trim+lower: Em processamento, Em trânsito, Enviado, Entregue", "cidade": "Cidade com nome canônico. Origem/regra: Dicionário de 20 cidades sobre trim+lower; ausente = NULL (R11)", "uf": "Sigla da UF. Origem/regra: Derivada da cidade; se a cidade for nula, usa o estado original se for sigla válida; senão NULL (R12)", "cliente": "Identificador do cliente. Origem/regra: trim", "cliente_cpf_mascarado": "CPF mascarado (***.***.***-DD). Origem/regra: Remove não dígitos; com 11 dígitos, mascara; senão NULL (R15)", "cliente_cpf_hash": "Hash SHA-256 do CPF com salt. Origem/regra: sha2(salt + 11 dígitos); permite contar CPFs distintos sem expor o CPF", "cliente_sexo": "Sexo padronizado (M, F, Outro). Origem/regra: Dicionário: 1 = M e 2 = F (premissa ISO 5218)", "cliente_telefone_mascarado": "Telefone mascarado ((DD) *****-NNNN). Origem/regra: Mascara os 5 primeiros dígitos; formato inválido = NULL", "cliente_email_mascarado": "E-mail mascarado (1ª letra + domínio). Origem/regra: lower+trim e mascara; inválido = NULL (R17)", "qtd_vendida": "Quantidade de unidades. Origem/regra: try_cast; nula ou <= 0 = erro R03 (quarentena)", "valor_unitario": "Preço unitário em R$. Origem/regra: try_cast; nulo ou <= 0 = erro R04 (quarentena)", "status_venda": "Status da venda padronizado. Origem/regra: Dicionário: Aprovada, Cancelada, Pendente, Desconhecido", "tipo_pagamento": "Forma de pagamento. Origem/regra: Cópia (valores já limpos na origem)", "avaliacao_produto": "Nota de 1 a 5. Origem/regra: try_cast; ausente permanece NULL (R14)", "desconto_aplicado": "Houve desconto. Origem/regra: Sim = true, Não = false (a partir de status_desconto)", "desconto_pct": "Percentual de desconto. Origem/regra: try_cast; fora de 0..100 = erro R05", "item_promocional": "Item promocional. Origem/regra: Sim = true, Não = false", "alertas": "IDs das regras de alerta violadas pelo registro. Origem/regra: Resultado das regras de severidade ALERTA", "_arquivo_origem": "Arquivo de origem. Origem/regra: Herdado da Bronze", "_id_lote_bronze": "Lote da Bronze que originou o registro. Origem/regra: Herdado de _id_lote", "_numero_linha_origem": "Identificador da linha na Bronze. Origem/regra: Herdado da Bronze", "_id_lote_silver": "Lote desta execução da Silver. Origem/regra: uuid4 por execução", "_data_processamento": "Momento do processamento. Origem/regra: current_timestamp()"}, "silver.vendas_quarentena": {"_numero_linha_origem": "Linha da Bronze rejeitada. Origem/regra: Chave para voltar à Bronze", "tipo_rejeicao": "ERRO_QUALIDADE ou DUPLICADA. Origem/regra: Definido pelo motivo da rejeição", "regras": "Regras violadas (R01..R05) ou DUP. Origem/regra: Regras de severidade ERRO; DUP para duplicata", "cliente": "Cliente do registro rejeitado. Origem/regra: Bronze", "produto_bruto": "Produto como veio. Origem/regra: Bronze", "data_venda_bruta": "Data da venda como veio. Origem/regra: Bronze", "qtd_vendida_bruta": "Quantidade como veio. Origem/regra: Bronze", "valor_unitario_bruto": "Preço unitário como veio. Origem/regra: Bronze", "_id_lote_silver": "Lote da Silver que rejeitou o registro. Origem/regra: uuid4 por execução", "_data_processamento": "Momento do processamento. Origem/regra: current_timestamp()"}, "gold.dim_tempo": {"sk_tempo": "Chave substituta no formato yyyyMMdd. Origem/regra: date_format(data, \"yyyyMMdd\")", "data": "Data do calendário. Origem/regra: Sequência diária de 01/12/2020 a 31/03/2024", "ano": "Ano. Origem/regra: year(data)", "trimestre": "Trimestre (1 a 4). Origem/regra: quarter(data)", "mes": "Mês (1 a 12). Origem/regra: month(data)", "nome_mes": "Nome do mês em português. Origem/regra: Lista fixa de nomes", "ano_mes": "Ano e mês (yyyy-MM). Origem/regra: date_format(data, \"yyyy-MM\")", "primeiro_dia_mes": "Primeiro dia do mês. Origem/regra: trunc(data, \"MM\")", "dia": "Dia do mês. Origem/regra: dayofmonth(data)", "dia_semana": "Dia da semana (1 = domingo). Origem/regra: dayofweek(data)", "nome_dia_semana": "Nome do dia da semana. Origem/regra: Lista fixa de nomes", "flag_fim_de_semana": "Sábado ou domingo. Origem/regra: dayofweek in (1, 7)"}, "gold.dim_produto": {"sk_produto": "Chave substituta do produto. Origem/regra: row_number() ordenado por produto", "produto": "Produto. Origem/regra: silver.vendas.produto", "tipo_produto": "Tipo de produto. Origem/regra: silver.vendas.tipo_produto"}, "gold.dim_localidade": {"sk_localidade": "Chave substituta da localidade. Origem/regra: row_number() ordenado por uf e cidade", "cidade": "Cidade (ou \"Não informada\"). Origem/regra: coalesce(cidade, \"Não informada\")", "uf": "UF (ou \"NI\"). Origem/regra: coalesce(uf, \"NI\")", "regiao": "Região do Brasil. Origem/regra: Mapa UF para região (Norte, Nordeste, Sudeste, Sul)"}, "gold.dim_pagamento": {"sk_pagamento": "Chave substituta da forma de pagamento. Origem/regra: row_number() ordenado por tipo_pagamento", "tipo_pagamento": "Forma de pagamento. Origem/regra: silver.vendas.tipo_pagamento"}, "gold.dim_status": {"sk_status": "Chave substituta da combinação de status. Origem/regra: row_number() ordenado por status_venda e status_envio", "status_venda": "Status da venda. Origem/regra: silver.vendas.status_venda", "status_envio": "Status do envio. Origem/regra: silver.vendas.status_envio"}, "gold.dim_cliente": {"sk_cliente": "Chave substituta do cliente. Origem/regra: row_number() ordenado por cliente", "cliente": "Identificador do cliente. Origem/regra: silver.vendas.cliente", "cliente_sexo": "Sexo do cliente. Origem/regra: Valor mais frequente do cliente; em empate, o mais recente", "qtd_sexos_distintos": "Quantidade de valores de sexo distintos do cliente. Origem/regra: countDistinct(cliente_sexo); 1 indica sem conflito", "qtd_vendas_historico": "Quantidade de vendas do cliente na Silver. Origem/regra: count(*) por cliente"}, "gold.fato_vendas": {"id_venda": "Identificador da venda (dimensão degenerada). Origem/regra: silver.vendas.id_venda", "sk_tempo_venda": "Chave de dim_tempo para a data da venda. Origem/regra: yyyyMMdd de data_venda", "sk_tempo_retirada": "Chave de dim_tempo para a data da retirada. Origem/regra: yyyyMMdd de data_retirada", "sk_produto": "Chave de dim_produto. Origem/regra: Junção por produto e tipo_produto", "sk_cliente": "Chave de dim_cliente. Origem/regra: Junção por cliente", "sk_localidade": "Chave de dim_localidade. Origem/regra: Junção por cidade e uf (nulos viram \"Não informada\" / \"NI\")", "sk_pagamento": "Chave de dim_pagamento. Origem/regra: Junção por tipo_pagamento", "sk_status": "Chave de dim_status. Origem/regra: Junção por status_venda e status_envio", "qtd_vendida": "Unidades vendidas. Origem/regra: silver.vendas.qtd_vendida", "valor_unitario": "Preço unitário em R$. Origem/regra: silver.vendas.valor_unitario", "valor_bruto": "Valor antes do desconto. Origem/regra: round(qtd_vendida * valor_unitario, 2)", "desconto_pct": "Percentual de desconto. Origem/regra: silver.vendas.desconto_pct", "valor_desconto": "Valor do desconto. Origem/regra: round(valor_bruto * desconto_pct / 100, 2)", "valor_liquido": "Valor após o desconto, sem frete. Origem/regra: round(valor_bruto * (1 - desconto_pct / 100), 2)", "valor_frete": "Valor do frete em R$ (NULL se ausente). Origem/regra: silver.vendas.valor_frete", "avaliacao_produto": "Nota do produto (NULL se ausente). Origem/regra: silver.vendas.avaliacao_produto", "desconto_aplicado": "Houve desconto. Origem/regra: silver.vendas.desconto_aplicado", "item_promocional": "Item promocional. Origem/regra: silver.vendas.item_promocional", "dias_ate_retirada": "Dias entre a venda e a retirada. Origem/regra: datediff; NULL se a retirada for anterior à venda", "flag_retirada_inconsistente": "Retirada anterior à data da venda. Origem/regra: data_retirada < data_venda", "flag_receita": "Conta como receita. Origem/regra: status_venda em STATUS_RECEITA (Aprovada)", "flag_demanda": "Conta como demanda de estoque. Origem/regra: status_venda em STATUS_DEMANDA (Aprovada, Pendente)", "qtd_alertas": "Quantidade de alertas de qualidade do registro. Origem/regra: size(alertas)", "_id_lote_silver": "Lote da Silver de origem. Origem/regra: Herdado da Silver", "_id_lote_gold": "Lote desta execução da Gold. Origem/regra: uuid4 por execução", "_data_processamento": "Momento do processamento. Origem/regra: current_timestamp()"}, "gold.agg_vendas_mensal_produto": {"primeiro_dia_mes": "Primeiro dia do mês de referência. Origem/regra: dim_tempo.primeiro_dia_mes", "ano_mes": "Ano e mês (yyyy-MM). Origem/regra: dim_tempo.ano_mes", "produto": "Produto. Origem/regra: dim_produto.produto", "tipo_produto": "Tipo de produto. Origem/regra: dim_produto.tipo_produto", "qtd_vendas": "Total de vendas. Origem/regra: count(*)", "qtd_vendas_aprovadas": "Vendas aprovadas. Origem/regra: Soma de 1 onde status_venda = Aprovada", "qtd_vendas_pendentes": "Vendas pendentes. Origem/regra: Soma de 1 onde status_venda = Pendente", "qtd_vendas_canceladas": "Vendas canceladas. Origem/regra: Soma de 1 onde status_venda = Cancelada", "qtd_vendas_desconhecidas": "Vendas com status desconhecido. Origem/regra: Soma de 1 onde status_venda = Desconhecido", "unidades_demanda": "Unidades de vendas aprovadas e pendentes (demanda de estoque). Origem/regra: Soma de qtd_vendida onde flag_demanda", "unidades_aprovadas": "Unidades de vendas aprovadas. Origem/regra: Soma de qtd_vendida onde flag_receita", "receita_bruta": "Valor bruto das vendas aprovadas. Origem/regra: Soma de valor_bruto onde flag_receita", "desconto_total": "Descontos das vendas aprovadas. Origem/regra: Soma de valor_desconto onde flag_receita", "receita_liquida": "Valor líquido das vendas aprovadas. Origem/regra: Soma de valor_liquido onde flag_receita", "frete_total": "Frete das vendas aprovadas (ignora nulos). Origem/regra: Soma de valor_frete onde flag_receita", "avaliacao_media": "Nota média das vendas não canceladas. Origem/regra: avg(avaliacao_produto) onde status_venda <> Cancelada", "ticket_medio": "Receita líquida por venda aprovada. Origem/regra: round(receita_liquida / qtd_vendas_aprovadas, 2)", "pct_cancelamento": "Percentual de vendas canceladas. Origem/regra: round(100 * canceladas / total, 2)"}, "gold.vw_vendas_analitica": {"id_venda": "Identificador da venda. Origem/regra: fato_vendas", "data_venda": "Data da venda. Origem/regra: dim_tempo (papel venda)", "ano": "Ano da venda. Origem/regra: dim_tempo", "trimestre": "Trimestre da venda. Origem/regra: dim_tempo", "mes": "Mês da venda. Origem/regra: dim_tempo", "nome_mes": "Nome do mês da venda. Origem/regra: dim_tempo", "ano_mes": "Ano e mês da venda. Origem/regra: dim_tempo", "data_retirada": "Data da retirada. Origem/regra: dim_tempo (papel retirada)", "produto": "Produto. Origem/regra: dim_produto", "tipo_produto": "Tipo de produto. Origem/regra: dim_produto", "cliente": "Cliente. Origem/regra: dim_cliente", "cliente_sexo": "Sexo do cliente. Origem/regra: dim_cliente", "cidade": "Cidade. Origem/regra: dim_localidade", "uf": "UF. Origem/regra: dim_localidade", "regiao": "Região. Origem/regra: dim_localidade", "tipo_pagamento": "Forma de pagamento. Origem/regra: dim_pagamento", "status_venda": "Status da venda. Origem/regra: dim_status", "status_envio": "Status do envio. Origem/regra: dim_status", "qtd_vendida": "Unidades vendidas. Origem/regra: fato_vendas", "valor_unitario": "Preço unitário. Origem/regra: fato_vendas", "valor_bruto": "Valor bruto. Origem/regra: fato_vendas", "desconto_pct": "Percentual de desconto. Origem/regra: fato_vendas", "valor_desconto": "Valor do desconto. Origem/regra: fato_vendas", "valor_liquido": "Valor líquido. Origem/regra: fato_vendas", "valor_frete": "Frete. Origem/regra: fato_vendas", "avaliacao_produto": "Nota do produto. Origem/regra: fato_vendas", "desconto_aplicado": "Houve desconto. Origem/regra: fato_vendas", "item_promocional": "Item promocional. Origem/regra: fato_vendas", "dias_ate_retirada": "Dias até a retirada. Origem/regra: fato_vendas", "flag_retirada_inconsistente": "Retirada anterior à venda. Origem/regra: fato_vendas", "flag_receita": "Conta como receita. Origem/regra: fato_vendas", "flag_demanda": "Conta como demanda de estoque. Origem/regra: fato_vendas", "qtd_alertas": "Quantidade de alertas de qualidade. Origem/regra: fato_vendas"}, "gold.previsao_demanda_mensal": {"produto": "Produto. Origem/regra: Série de agg_vendas_mensal_produto", "mes_referencia": "Mês previsto (primeiro dia). Origem/regra: Meses seguintes ao último mês do histórico", "unidades_previstas": "Demanda prevista em unidades. Origem/regra: Modelo escolhido no backtest (média histórica)", "limite_inferior_80": "Limite inferior do intervalo de 80%. Origem/regra: previsto - 1,2816 * desvio do erro (mínimo 0)", "limite_superior_80": "Limite superior do intervalo de 80%. Origem/regra: previsto + 1,2816 * desvio do erro", "modelo": "Modelo utilizado. Origem/regra: Escolhido pelo critério de parcimônia", "id_lote": "Lote da execução. Origem/regra: uuid4 por execução", "data_processamento": "Momento do processamento. Origem/regra: Horário da execução"}, "gold.recomendacao_estoque": {"produto": "Produto. Origem/regra: Série de agg_vendas_mensal_produto", "periodo_inicio": "Primeiro mês coberto. Origem/regra: Primeiro mês previsto", "periodo_fim": "Último mês coberto. Origem/regra: Último mês previsto", "demanda_prevista_3m": "Demanda prevista para os 3 meses. Origem/regra: Soma das previsões mensais", "estoque_seguranca": "Estoque de segurança. Origem/regra: z(95%) * desvio do erro mensal * raiz de 3", "estoque_recomendado": "Necessidade de estoque para o período. Origem/regra: ceil(demanda_prevista_3m + estoque_seguranca)", "nivel_servico": "Nível de serviço adotado. Origem/regra: Parâmetro NIVEL_SERVICO (0,95)", "desvio_erro_mensal": "Desvio do erro de previsão mensal. Origem/regra: s * raiz(1 + 1/n) sobre os 36 meses (média histórica)", "modelo": "Modelo utilizado. Origem/regra: Escolhido pelo critério de parcimônia", "id_lote": "Lote da execução. Origem/regra: uuid4 por execução", "data_processamento": "Momento do processamento. Origem/regra: Horário da execução"}, "gold.backtest_modelos": {"modelo": "Modelo avaliado. Origem/regra: Lista de candidatos do notebook 07", "MAE": "Erro absoluto médio (unidades). Origem/regra: Backtest com origem expansiva, 10 cortes x 3 meses", "RMSE": "Raiz do erro quadrático médio (unidades). Origem/regra: Backtest", "WAPE_pct": "Erro percentual ponderado. Origem/regra: 100 * soma(erro absoluto) / soma(real)", "vies_pct": "Viés percentual (negativo = subestima). Origem/regra: 100 * soma(previsto - real) / soma(real)", "status": "ESCOLHIDO, equivalente ou pior. Origem/regra: Critério de parcimônia (tolerância de 0,5 p.p. de WAPE)"}, "ops.log_execucao": {"id_execucao": "Identificador da execução do orquestrador. Origem/regra: uuid4 por execução", "ordem": "Ordem da etapa. Origem/regra: Lista ETAPAS do orquestrador", "etapa": "bronze, silver, gold ou previsao. Origem/regra: Lista ETAPAS", "notebook": "Notebook chamado. Origem/regra: Lista ETAPAS", "status": "SUCESSO, GATE_REPROVADO, FALHA ou IGNORADA. Origem/regra: Resultado da etapa e do gate", "inicio": "Início da etapa. Origem/regra: Relógio do orquestrador", "fim": "Fim da etapa. Origem/regra: Relógio do orquestrador", "duracao_s": "Duração em segundos. Origem/regra: fim - inicio", "id_lote_camada": "Lote da camada (liga ao log_pipeline e às colunas _id_lote_*). Origem/regra: Retorno do notebook", "qtd_lida": "Registros lidos. Origem/regra: Retorno do notebook", "qtd_gravada": "Registros gravados. Origem/regra: Retorno do notebook", "qtd_rejeitada": "Registros em quarentena. Origem/regra: Retorno do notebook", "qtd_duplicada": "Duplicatas removidas. Origem/regra: Retorno do notebook", "detalhe": "Mensagem do gate ou erro. Origem/regra: Gate ou exceção"}, "ops.log_pipeline": {"id_lote": "Lote da camada. Origem/regra: uuid4 por execução do notebook", "etapa": "bronze, silver ou gold. Origem/regra: Notebook", "tabela_destino": "Tabela gravada. Origem/regra: Notebook", "qtd_lida": "Registros lidos. Origem/regra: Contagem", "qtd_gravada": "Registros gravados. Origem/regra: Contagem", "status": "SUCESSO ou DIVERGENCIA. Origem/regra: lidos = gravados (Bronze); reconciliação (Silver)", "inicio": "Início. Origem/regra: Relógio do notebook", "fim": "Fim. Origem/regra: Relógio do notebook", "qtd_rejeitada": "Registros em quarentena (Silver). Origem/regra: Contagem", "qtd_duplicada": "Duplicatas removidas (Silver). Origem/regra: Contagem"}, "ops.qualidade_resultados": {"id_lote": "Lote da camada. Origem/regra: uuid4 por execução do notebook", "etapa": "silver ou gold. Origem/regra: Notebook", "regra_id": "Identificador da regra (R01 a R19, G01 a G06). Origem/regra: Catálogo de regras", "descricao": "Descrição da regra. Origem/regra: Catálogo de regras", "severidade": "ERRO ou ALERTA. Origem/regra: Catálogo de regras", "qtd_afetada": "Registros que violam a regra. Origem/regra: Contagem", "qtd_total": "Registros avaliados. Origem/regra: Contagem", "pct_afetado": "Percentual afetado. Origem/regra: 100 * qtd_afetada / qtd_total", "data_execucao": "Momento da medição. Origem/regra: Relógio do notebook"}}''')
VIEWS = ["gold.vw_vendas_analitica", "ops.vw_ultima_execucao", "ops.vw_historico_execucoes", "ops.vw_funil_registros", "ops.vw_qualidade_ultima", "ops.vw_score_qualidade"]

sql(f"COMMENT ON CATALOG {CATALOGO} IS 'PoC de vendas de 3 produtos em arquitetura medalhão (Bronze, Silver, Gold) com observabilidade em ops'", "comentário do catálogo")
for schema, texto in {"bronze": "Dado bruto fiel à origem, com auditoria. Contém dados pessoais em claro (acesso restrito)",
                      "silver": "Dado tratado e padronizado, sem dados pessoais em claro, mais a quarentena",
                      "gold": "Camada analítica: esquema estrela, agregações, previsão e recomendação de estoque. Sem dados pessoais",
                      "ops": "Observabilidade: logs de execução, resultados de qualidade e views de painel"}.items():
    sql(f"COMMENT ON SCHEMA {CATALOGO}.{schema} IS '{texto}'", f"comentário do schema {schema}")

for tabela, texto in COMENTARIOS_TABELAS.items():
    sql(f"COMMENT ON TABLE {CATALOGO}.{tabela} IS '{texto}'", f"comentário da tabela {tabela}")

for tabela, colunas in COMENTARIOS_COLUNAS.items():
    for coluna, texto in colunas.items():
        if tabela in VIEWS:
            sql(f"COMMENT ON COLUMN {CATALOGO}.{tabela}.{coluna} IS '{texto}'", f"comentário da coluna {tabela}.{coluna}")
        else:
            sql(f"ALTER TABLE {CATALOGO}.{tabela} ALTER COLUMN {coluna} COMMENT '{texto}'", f"comentário da coluna {tabela}.{coluna}")

## 2. Tags de governança (camada, domínio, sensibilidade)

In [ ]:
tabelas_dados = [t for t in COMENTARIOS_TABELAS if t not in VIEWS]
for t in tabelas_dados:
    camada = t.split(".")[0]
    pii = "true" if t == "bronze.vendas_raw" else "false"
    sql(f"ALTER TABLE {CATALOGO}.{t} SET TAGS ('camada' = '{camada}', 'dominio' = 'vendas', 'contem_pii' = '{pii}')", f"tags da tabela {t}")

for coluna in ["cliente_cpf", "cliente_telefone", "cliente_email"]:
    sql(f"ALTER TABLE {CATALOGO}.bronze.vendas_raw ALTER COLUMN {coluna} SET TAGS ('pii' = 'true', 'classificacao' = 'confidencial')", f"tag PII bronze.{coluna}")
for coluna, classe in {"cliente_cpf_mascarado": "mascarado", "cliente_telefone_mascarado": "mascarado",
                       "cliente_email_mascarado": "mascarado", "cliente_cpf_hash": "pseudonimizado"}.items():
    sql(f"ALTER TABLE {CATALOGO}.silver.vendas ALTER COLUMN {coluna} SET TAGS ('pii' = 'derivado', 'classificacao' = '{classe}')", f"tag PII silver.{coluna}")

## 3. Chaves primárias e estrangeiras (informativas)
Não são impostas pelo Delta, mas documentam o modelo, aparecem no Catalog Explorer e ajudam o Genie a entender as junções.

In [ ]:
PKS = {"dim_tempo": "sk_tempo", "dim_produto": "sk_produto", "dim_localidade": "sk_localidade", "dim_pagamento": "sk_pagamento",
       "dim_status": "sk_status", "dim_cliente": "sk_cliente", "fato_vendas": "id_venda"}
for tabela, coluna in PKS.items():
    sql(f"ALTER TABLE {CATALOGO}.gold.{tabela} ALTER COLUMN {coluna} SET NOT NULL", f"NOT NULL {tabela}.{coluna}")
    sql(f"ALTER TABLE {CATALOGO}.gold.{tabela} ADD CONSTRAINT pk_{tabela} PRIMARY KEY ({coluna})", f"PK {tabela}")

FKS = [("sk_tempo_venda", "dim_tempo", "sk_tempo", "tempo_venda"), ("sk_tempo_retirada", "dim_tempo", "sk_tempo", "tempo_retirada"),
       ("sk_produto", "dim_produto", "sk_produto", "produto"), ("sk_cliente", "dim_cliente", "sk_cliente", "cliente"),
       ("sk_localidade", "dim_localidade", "sk_localidade", "localidade"), ("sk_pagamento", "dim_pagamento", "sk_pagamento", "pagamento"),
       ("sk_status", "dim_status", "sk_status", "status")]
for col, dim, pk, nome in FKS:
    sql(f"ALTER TABLE {CATALOGO}.gold.fato_vendas ADD CONSTRAINT fk_fato_{nome} FOREIGN KEY ({col}) REFERENCES {CATALOGO}.gold.{dim} ({pk})", f"FK fato_vendas.{col}")

## 4. Máscara de coluna (Unity Catalog) nos dados pessoais da Bronze
Quem não é o dono do pipeline nem membro do grupo autorizado vê só os 2 primeiros caracteres. O dono precisa continuar vendo o dado em claro,
senão a Silver passaria a ler valores mascarados e o hash do CPF seria calculado sobre dado errado.

In [ ]:
sql(f"""
CREATE OR REPLACE FUNCTION {CATALOGO}.bronze.mascarar_pii(valor STRING)
RETURNS STRING
COMMENT 'Mascara dado pessoal: exibe o valor completo apenas ao dono do pipeline e ao grupo autorizado'
RETURN CASE WHEN current_user() = '{DONO}' OR is_account_group_member('{GRUPO_PII}') THEN valor
            ELSE CONCAT(LEFT(valor, 2), '***') END
""", "função mascarar_pii")
for coluna in ["cliente_cpf", "cliente_telefone", "cliente_email"]:
    sql(f"ALTER TABLE {CATALOGO}.bronze.vendas_raw ALTER COLUMN {coluna} SET MASK {CATALOGO}.bronze.mascarar_pii", f"máscara bronze.{coluna}")

## 5. Concessões de acesso (modelo de perfis)
Engenharia: todas as camadas. Analistas, BI e Genie: somente a Gold. Bronze e Silver ficam sem concessão para os demais usuários.

In [ ]:
if EXECUTAR_GRANTS:
    sql(f"GRANT USE CATALOG ON CATALOG {CATALOGO} TO `account users`", "grant uso do catálogo")
    sql(f"GRANT USE SCHEMA, SELECT ON SCHEMA {CATALOGO}.gold TO `account users`", "grant leitura da Gold")
else:
    resultado.append(("grants da Gold", "NÃO EXECUTADO", "EXECUTAR_GRANTS = False"))

## 6. Resumo e verificações

In [ ]:
import pandas as pd
res = pd.DataFrame(resultado, columns=["acao", "status", "detalhe"])
print(res["status"].value_counts().to_string())
falhas = res[res["status"] == "FALHA"]
if falhas.empty:
    print("Nenhuma falha registrada.")
else:
    display(falhas)

In [ ]:
for consulta in [f"SELECT table_schema, table_name, tag_name, tag_value FROM {CATALOGO}.information_schema.table_tags ORDER BY 1, 2",
                 f"SELECT table_schema, table_name, column_name, tag_name, tag_value FROM {CATALOGO}.information_schema.column_tags ORDER BY 1, 2, 3",
                 f"SELECT table_name, constraint_name, constraint_type FROM {CATALOGO}.information_schema.table_constraints WHERE table_schema = 'gold' ORDER BY 1, 2"]:
    try:
        display(spark.sql(consulta))
    except Exception as e:
        print("Consulta não disponível:", str(e)[:150])

display(spark.sql(f"SELECT cliente_cpf, cliente_telefone, cliente_email FROM {CATALOGO}.bronze.vendas_raw LIMIT 3"))

In [ ]:
dbutils.notebook.exit(json.dumps({"id_lote": "governanca", "qtd_lida": len(res), "qtd_gravada": int((res["status"].str.startswith("OK")).sum()),
                                  "qtd_rejeitada": int((res["status"] == "FALHA").sum()), "qtd_duplicada": 0,
                                  "status": "SUCESSO" if (res["status"] == "FALHA").sum() == 0 else "DIVERGENCIA"}))

In [ ]:
CATALOGO = "desafio_vendas"
consultas = {
  "tags de tabela": f"SELECT schema_name, table_name, tag_name, tag_value FROM {CATALOGO}.information_schema.table_tags ORDER BY 1, 2",
  "tags de coluna": f"SELECT schema_name, table_name, column_name, tag_name, tag_value FROM {CATALOGO}.information_schema.column_tags ORDER BY 1, 2, 3",
  "máscaras aplicadas": f"SELECT * FROM {CATALOGO}.information_schema.column_masks",
}
for nome, q in consultas.items():
    print("==", nome)
    try: display(spark.sql(q))
    except Exception as e: print("não disponível:", str(e)[:150])

# O que um usuário SEM autorização veria (mesma regra da máscara, aplicada manualmente)
display(spark.sql(f"""SELECT CONCAT(LEFT(cliente_cpf,2),'***') AS cpf_visto_por_outro_usuario,
                             CONCAT(LEFT(cliente_email,2),'***') AS email_visto_por_outro_usuario
                      FROM {CATALOGO}.bronze.vendas_raw LIMIT 3"""))